# 15 — Milestone 2: parse every paper on Drive

Owner: Member 1. **The notebook to open on Google Colab (CPU runtime) for Milestone 2.** It loads `00_config`,
`02_download` (only for its run lock and its receipts) and `13_dedup_export` (which loads `10`–`12` when they
exist, see the contract in `13`), then:

1. **Papers**: the IDs of `state/discovery_manifest.json` that exist on arXiv, not the folders of `data/raw/`
   (it also holds `.staging`, the downloader's work folder). `TEST_PAPERS` takes that many, spread over the
   four ranges, plus the hard cases (several roots, `\begin{document}` in an included file, no section, many
   versions, `\chapter`, missing `.bbl`).
2. **Parse in parallel**: `WORKERS` processes run `parse_paper` and write `data/parsed/<id>.json` (one file
   per paper with all its versions; the raw folders of Milestone 1 are only read); the main process appends `logs/m2_parse_status.csv` (one row per version) and the resume index
   `state/m2/done.jsonl`. A paper that fails is logged as `error` and does not stop the run.
3. **Tables**: the `data/parsed/<id>.json` of every paper → `data/m2/elements.parquet`, `hierarchy.parquet`,
   `bibtex.parquet`, written 500 papers at a time with `pyarrow` (never the whole corpus in RAM), with the
   invariant checks.
4. **Statistics and benchmark**: `logs/m2_stats.json` (for the report) and `logs/m2_benchmark.json` over
   every session.

**Resume.** After a disconnect, run all cells again: the papers in `state/m2/done.jsonl` with the current
`PARSER_VERSION` are skipped (one file read instead of 20,000 files of `data/parsed/` on Drive). Each session saves its
time and RAM to `logs/m2_sessions/` every 5 minutes, so a session that Colab kills still counts.

**Before the full run**: set `TEST_PAPERS = None` and `FRESH_RUN = True` once (the test's logs, sessions,
`data/parsed/` and tables are deleted, so the benchmark and the output size cover the full run only), then `FRESH_RUN = False` for the next sessions.

## 0. Setup

`WORKERS` processes parse in parallel (`multiprocessing`, `fork` on Colab's Linux, so the functions defined
in the notebooks work in the workers). Reading from Drive is the bottleneck, not the CPU, so more workers than
Colab's 2 cores still help; 4–8 is the range to try. Without `fork` (Windows) the papers are parsed one by
one. `M2_ROOT` points to another project folder with `data/raw`, `state` and `logs` (e.g. a local copy or
`test_runs/1000_versions`); another notebook can set `M2_OVERRIDES = {...}` before `%run -i` to change any
value of this cell.

In [ ]:
# On Colab: clone or update the branch, install packages and work from src/ (the notebooks use relative paths).
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    # Every run: the notebooks loaded with %run must be the latest commit of the branch.
    REPO = Path("/content/intro-ds-arxiv-project")
    BRANCH = "m2-pipeline"   # Milestone 2 branch (10-13 + 15 together), "main" once merged
    URL = "https://github.com/thinhelzan842/intro-ds-arxiv-project.git"
    if not (REPO / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPO)], check=True)
    else:   # explicit refspec: also works in a clone of another single branch (e.g. VA, cloned by 05)
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin",
                        f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "src/requirements.txt")], check=True)
    os.chdir(REPO / "src")
    print("Code in use:", subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"],
                                         capture_output=True, text=True).stdout.strip())

RUN_TEST = False          # True also runs the offline tests of 02 and 13
TEST_PAPERS = 100         # None: every paper; N: N papers spread over the four ranges, plus HARD_CASES
HARD_CASES = ("2609-32839", "2609-32858", "2603-21039", "2609-33000", "2605-22101", "2609-33059",
              "2603-22481", "2603-20714", "2603-23413")
WORKERS = 6               # parsing processes (4-8: Drive reads are the bottleneck, not the 2 CPU cores)
FRESH_RUN = False         # True: delete the M2 logs, sessions, index, data/parsed/ and tables first
CLEAN_OLD_PARSED = False  # True once: delete data/raw/<id>/parsed.json written before 19 Nov (now data/parsed/)
PAPER_TIMEOUT = 600       # seconds for one paper; past it the paper is an error ("timeout") and later sessions skip it
RETRY_FAILED = False      # True: parse again the papers that timed out or lost their worker in an earlier session
CARRY_OVER = False        # True after 13.7 -> 13.8: keep the 13.7 results 13.8 cannot change (section 1)
READ_TIMEOUT = 600        # seconds to read 500 files of data/parsed/ when the tables are built (Drive may hang)
MERGE_TABLES = True       # build the three Parquet tables at the end
M2_ROOT = None            # another folder with data/raw, state, logs (default: PROJECT_ROOT)
# TEAM_USE = ()           # uncomment: the fallback of 13 alone, without notebooks 10-12 (to compare)
FLUSH_DRIVE_AT_END = True  # flush and unmount Google Drive at the end, so every file is saved

globals().update(globals().get("M2_OVERRIDES", {}))

In [ ]:
get_ipython().run_line_magic("run", "-i ./00_config.ipynb")
if M2_ROOT:
    # Another project folder: move every path before 02 and 13 derive theirs from them.
    DATA_DIR, LOG_DIR, STATE_DIR = Path(M2_ROOT) / "data" / "raw", Path(M2_ROOT) / "logs", Path(M2_ROOT) / "state"
    PAPER_LOG = LOG_DIR / "papers.csv"
    for _folder in (LOG_DIR, STATE_DIR):
        _folder.mkdir(parents=True, exist_ok=True)
    print(f"M2_ROOT: {M2_ROOT}")

for notebook in ("02_download", "13_dedup_export"):
    get_ipython().run_line_magic("run", f"-i ./{notebook}.ipynb")

import contextlib
import multiprocessing
import platform
import shutil
import signal
import statistics
import tempfile
import threading
import uuid
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import psutil
import pyarrow as pa
import pyarrow.parquet as pq

M2_DATA = DATA_DIR.parent / "m2"                    # data/m2/*.parquet
PARSED_DIR = DATA_DIR.parent / "parsed"            # data/parsed/<id>.json, one per paper (all versions)
M2_STATE = STATE_DIR / "m2"
DONE_INDEX = M2_STATE / "done.jsonl"                # resume index: one line per parsed paper
FAILED_INDEX = M2_STATE / "failed.jsonl"            # papers that timed out or lost their worker
STATUS_LOG = LOG_DIR / "m2_parse_status.csv"        # one row per version
STATUS_FIELDS = ["arxiv_id", "version", "status", "bib_source", "n_nodes", "seconds", "error"]
M2_SESSIONS = LOG_DIR / "m2_sessions"               # one record per Colab session
M2_SESSIONS_LOG = LOG_DIR / "m2_benchmark_sessions.jsonl"
M2_RAM_LOG = LOG_DIR / "m2_ram_timeseries.csv"
M2_BENCHMARK = LOG_DIR / "m2_benchmark.json"
M2_STATS = LOG_DIR / "m2_stats.json"
MB = 1024 * 1024

if FRESH_RUN:
    for path in (STATUS_LOG, M2_SESSIONS_LOG, M2_RAM_LOG, M2_BENCHMARK, M2_STATS):
        path.unlink(missing_ok=True)
    for folder in (M2_SESSIONS, M2_STATE, M2_DATA, PARSED_DIR):
        shutil.rmtree(folder, ignore_errors=True)
    print("FRESH_RUN: M2 logs, sessions, index, data/parsed/ and tables deleted")
M2_STATE.mkdir(parents=True, exist_ok=True)
PARSED_DIR.mkdir(parents=True, exist_ok=True)
if CLEAN_OLD_PARSED:   # before 19 Nov, parsed.json was written inside each raw paper folder
    _old = [path for path in DATA_DIR.glob("*/parsed.json") if path.is_file()]
    for _path in _old:
        _path.unlink()
    print(f"CLEAN_OLD_PARSED: {len(_old):,} data/raw/<id>/parsed.json deleted")


def parsed_path(folder_id):
    """data/parsed/<folder_id>.json: to_parsed_json (13) of one paper, every version."""
    return PARSED_DIR / f"{folder_id}.json"


print(f"Parser {PARSER_VERSION} | workers {WORKERS} | data {DATA_DIR}")
print("Implementation: " + ", ".join(f"{name}={source[:2]}" for name, source in IMPLEMENTATION.items()))

## 1. Papers to parse

The IDs come from the discovery manifest (papers with at least one version); without a manifest, from the
folders of `data/raw/` whose name is an ID. The folders that are not an assigned paper are printed (on Drive:
`.staging`, the downloader's work folder, which makes 20,001 folders for 20,000 IDs). `versions` here is the
count of `metadata.json` / the manifest, never the `vN/` folders.

**13.7 → 13.8 (22 Nov).** The new version can change a paper only if one of its `.tex` files has a carriage
return (10) or a nested, unclosed or `\input`-holding macro definition (11). `CARRY_OVER = True` keeps the
13.7 results of the other papers instead of parsing all 20,000 again: it parses the papers of `REPARSE_13_8`
and a random sample of the others, whose new files must equal their 13.7 files (checked in section 3).

In [ ]:
# 13.7 -> 13.8 (22 Nov): 10 reads CR-only and CRLF files as LF (2603.21544 and 2605.18061 had no root), and 11's
# expand_macros no longer cuts the text that follows a nested definition. 13.8 can change the output of a paper
# only if one of its .tex files has a carriage return, or a macro definition that holds another definition, is
# not closed in its file, or holds an \input: REPARSE_13_8 lists the 1,775 papers with such a file (found
# from the files alone, so the list holds on Colab too). On a local copy of data/raw, the 13.7 and 13.8 outputs
# differ for 436 papers, all in the list. CARRY_OVER = True keeps the 13.7 results of the other papers and
# parses again the list plus CARRY_SAMPLE other papers, whose new files must equal their 13.7 files (section 3).
import hashlib
import random

CARRY_FROM = "13.7|10:team|11:team|12:team"
CARRY_TO = "13.8|10:team|11:team|12:team"
CARRY_SAMPLE = 300
REPARSE_13_8 = frozenset("""
2603-20585 2603-20596 2603-20605 2603-20609 2603-20631 2603-20633 2603-20643 2603-20644 2603-20651 2603-20669
2603-20675 2603-20677 2603-20686 2603-20692 2603-20698 2603-20699 2603-20707 2603-20708 2603-20717 2603-20720
2603-20728 2603-20747 2603-20755 2603-20770 2603-20774 2603-20776 2603-20794 2603-20814 2603-20830 2603-20849
2603-20852 2603-20859 2603-20864 2603-20865 2603-20889 2603-20890 2603-20934 2603-20935 2603-20973 2603-21000
2603-21002 2603-21007 2603-21026 2603-21028 2603-21045 2603-21046 2603-21052 2603-21057 2603-21075 2603-21087
2603-21104 2603-21113 2603-21121 2603-21151 2603-21159 2603-21169 2603-21180 2603-21184 2603-21191 2603-21196
2603-21204 2603-21208 2603-21214 2603-21230 2603-21238 2603-21239 2603-21242 2603-21251 2603-21253 2603-21258
2603-21261 2603-21266 2603-21287 2603-21290 2603-21308 2603-21324 2603-21325 2603-21346 2603-21348 2603-21356
2603-21361 2603-21363 2603-21379 2603-21387 2603-21394 2603-21404 2603-21429 2603-21432 2603-21452 2603-21462
2603-21464 2603-21481 2603-21484 2603-21532 2603-21535 2603-21544 2603-21555 2603-21576 2603-21597 2603-21638
2603-21640 2603-21646 2603-21650 2603-21667 2603-21668 2603-21672 2603-21702 2603-21713 2603-21717 2603-21743
2603-21751 2603-21752 2603-21757 2603-21774 2603-21780 2603-21783 2603-21805 2603-21839 2603-21855 2603-21857
2603-21861 2603-21873 2603-21877 2603-21880 2603-21893 2603-21909 2603-21910 2603-21917 2603-21919 2603-21926
2603-21929 2603-21933 2603-21934 2603-21935 2603-21936 2603-21939 2603-21942 2603-21947 2603-21965 2603-21992
2603-22001 2603-22011 2603-22032 2603-22033 2603-22045 2603-22048 2603-22058 2603-22063 2603-22070 2603-22072
2603-22076 2603-22084 2603-22094 2603-22107 2603-22109 2603-22125 2603-22128 2603-22132 2603-22156 2603-22166
2603-22171 2603-22183 2603-22188 2603-22194 2603-22196 2603-22202 2603-22203 2603-22220 2603-22233 2603-22248
2603-22249 2603-22261 2603-22264 2603-22273 2603-22279 2603-22284 2603-22298 2603-22326 2603-22337 2603-22352
2603-22354 2603-22364 2603-22373 2603-22381 2603-22404 2603-22408 2603-22427 2603-22435 2603-22438 2603-22446
2603-22449 2603-22454 2603-22462 2603-22485 2603-22498 2603-22517 2603-22533 2603-22540 2603-22541 2603-22543
2603-22553 2603-22556 2603-22561 2603-22644 2603-22650 2603-22651 2603-22659 2603-22661 2603-22662 2603-22695
2603-22713 2603-22741 2603-22762 2603-22764 2603-22788 2603-22809 2603-22814 2603-22816 2603-22826 2603-22827
2603-22828 2603-22835 2603-22864 2603-22877 2603-22890 2603-22896 2603-22902 2603-22908 2603-22909 2603-22922
2603-22923 2603-22925 2603-22940 2603-22947 2603-22982 2603-22990 2603-23006 2603-23046 2603-23051 2603-23060
2603-23074 2603-23077 2603-23087 2603-23092 2603-23096 2603-23108 2603-23115 2603-23121 2603-23135 2603-23137
2603-23164 2603-23184 2603-23212 2603-23224 2603-23240 2603-23250 2603-23288 2603-23298 2603-23302 2603-23304
2603-23315 2603-23355 2603-23360 2603-23366 2603-23378 2603-23385 2603-23388 2603-23398 2603-23401 2603-23402
2603-23422 2603-23453 2603-23454 2603-23461 2603-23462 2603-23463 2603-23472 2603-23488 2603-23497 2603-23504
2603-23545 2603-23548 2603-23560 2603-23579 2603-23586 2603-23588 2603-23618 2603-23621 2603-23630 2603-23637
2603-23647 2603-23649 2603-23653 2603-23664 2603-23671 2603-23692 2603-23726 2603-23731 2603-23744 2603-23758
2603-23776 2603-23778 2603-23822 2603-23827 2603-23828 2603-23851 2603-23866 2603-23876 2603-23881 2603-23894
2603-23915 2603-23921 2603-23932 2603-23936 2603-23957 2603-23962 2603-23982 2603-24018 2603-24028 2603-24032
2603-24112 2603-24123 2603-24140 2603-24153 2603-24159 2603-24171 2603-24176 2603-24186 2603-24187 2603-24188
2603-24191 2603-24197 2603-24227 2603-24235 2603-24255 2603-24280 2603-24290 2603-24310 2603-24314 2603-24316
2603-24318 2603-24331 2603-24332 2603-24333 2603-24335 2603-24337 2603-24368 2603-24384 2603-24406 2603-24433
2603-24443 2603-24452 2603-24457 2603-24458 2603-24463 2603-24468 2603-24469 2603-24479 2603-24499 2603-24514
2603-24523 2603-24540 2603-24545 2603-24548 2603-24569 2603-24570 2603-24578 2603-24586 2603-24605 2603-24617
2603-24618 2603-24627 2603-24632 2603-24645 2603-24657 2603-24672 2603-24690 2603-24692 2603-24700 2603-24707
2603-24782 2603-24789 2603-24794 2603-24795 2603-24803 2603-24804 2603-24833 2603-24836 2603-24851 2603-24881
2603-24890 2603-24900 2603-24903 2603-24905 2603-24906 2603-24919 2603-24942 2603-24944 2603-24951 2603-24960
2603-24972 2603-24990 2603-25007 2603-25012 2603-25017 2603-25029 2603-25032 2603-25040 2603-25043 2603-25055
2603-25057 2603-25059 2603-25069 2603-25082 2603-25120 2603-25132 2603-25134 2603-25148 2603-25149 2603-25151
2603-25155 2603-25174 2603-25177 2603-25193 2603-25215 2603-25231 2603-25235 2603-25240 2603-25265 2603-25270
2603-25274 2603-25277 2603-25282 2603-25286 2603-25288 2603-25306 2603-25337 2603-25355 2603-25357 2603-25358
2603-25373 2603-25380 2603-25395 2603-25410 2603-25412 2603-25417 2603-25444 2603-25456 2603-25458 2603-25461
2603-25466 2603-25474 2603-25483 2603-25486 2603-25489 2603-25495 2603-25506 2603-25507 2603-25513 2603-25515
2603-25528 2603-25542 2603-25545 2603-25547 2603-25552 2603-25553 2603-25554 2603-25567 2603-25569 2605-07348
2605-07350 2605-07355 2605-07361 2605-07362 2605-07368 2605-07371 2605-07373 2605-07394 2605-07411 2605-07444
2605-07460 2605-07470 2605-07471 2605-07474 2605-07504 2605-07507 2605-07516 2605-07518 2605-07528 2605-07530
2605-07534 2605-07547 2605-07565 2605-07569 2605-07570 2605-07571 2605-07615 2605-07626 2605-07636 2605-07672
2605-07675 2605-07679 2605-07703 2605-07721 2605-07730 2605-07736 2605-07753 2605-07799 2605-07800 2605-07801
2605-07820 2605-07824 2605-07834 2605-07840 2605-07852 2605-07882 2605-07898 2605-07899 2605-07914 2605-07924
2605-07925 2605-07943 2605-07967 2605-07996 2605-08029 2605-08047 2605-08048 2605-08054 2605-08072 2605-08076
2605-08078 2605-08081 2605-08087 2605-08118 2605-08151 2605-08153 2605-08166 2605-08172 2605-08211 2605-08234
2605-08235 2605-08240 2605-08246 2605-08254 2605-08304 2605-08339 2605-08355 2605-08366 2605-08367 2605-08375
2605-08378 2605-08381 2605-08394 2605-08401 2605-08411 2605-08429 2605-08431 2605-08451 2605-08472 2605-08473
2605-08487 2605-08491 2605-08495 2605-08509 2605-08538 2605-08548 2605-08566 2605-08579 2605-08590 2605-08596
2605-08599 2605-08607 2605-08610 2605-08616 2605-08625 2605-08656 2605-08661 2605-08669 2605-08675 2605-08678
2605-08684 2605-08696 2605-08699 2605-08706 2605-08709 2605-08719 2605-08745 2605-08748 2605-08751 2605-08778
2605-08780 2605-08789 2605-08793 2605-08796 2605-08813 2605-08818 2605-08822 2605-08824 2605-08825 2605-08879
2605-08884 2605-08893 2605-08899 2605-08900 2605-08904 2605-08916 2605-08921 2605-08926 2605-08928 2605-08933
2605-08946 2605-08957 2605-08974 2605-08976 2605-08981 2605-08983 2605-09006 2605-09008 2605-09010 2605-09029
2605-09063 2605-09064 2605-09073 2605-09079 2605-09090 2605-09103 2605-09105 2605-09117 2605-09122 2605-09151
2605-09161 2605-09173 2605-09191 2605-09216 2605-09234 2605-09235 2605-09246 2605-09250 2605-09261 2605-09282
2605-09286 2605-09314 2605-09321 2605-09383 2605-09390 2605-09395 2605-09406 2605-09421 2605-09423 2605-09451
2605-09466 2605-09474 2605-09478 2605-09503 2605-09509 2605-09512 2605-09524 2605-09527 2605-09554 2605-09557
2605-09577 2605-09585 2605-09605 2605-09612 2605-09647 2605-09653 2605-09721 2605-09735 2605-09743 2605-09748
2605-09766 2605-09767 2605-09778 2605-09801 2605-09805 2605-09807 2605-09828 2605-09847 2605-09854 2605-09859
2605-09860 2605-09900 2605-09910 2605-09930 2605-09931 2605-09932 2605-09954 2605-09959 2605-09972 2605-09988
2605-10017 2605-10041 2605-10048 2605-10052 2605-10068 2605-10072 2605-10074 2605-10075 2605-10081 2605-10090
2605-10113 2605-10155 2605-10164 2605-10173 2605-10187 2605-10202 2605-10205 2605-10224 2605-10248 2605-10258
2605-10267 2605-10284 2605-10290 2605-10298 2605-10307 2605-10309 2605-10353 2605-10365 2605-10369 2605-10376
2605-10379 2605-10387 2605-10388 2605-10399 2605-10400 2605-10413 2605-10417 2605-10421 2605-10428 2605-10452
2605-10474 2605-10479 2605-10486 2605-10506 2605-10532 2605-10543 2605-10553 2605-10580 2605-10585 2605-10600
2605-10601 2605-10609 2605-10612 2605-10617 2605-10625 2605-10638 2605-10656 2605-10706 2605-10760 2605-10766
2605-10768 2605-10770 2605-10777 2605-10804 2605-10806 2605-10811 2605-10813 2605-10840 2605-10841 2605-10883
2605-10888 2605-10889 2605-10904 2605-10912 2605-10915 2605-10936 2605-10945 2605-10977 2605-10983 2605-10988
2605-11008 2605-11009 2605-11011 2605-11013 2605-11021 2605-11026 2605-11061 2605-11065 2605-11076 2605-11097
2605-11106 2605-11141 2605-11163 2605-11180 2605-11182 2605-11199 2605-11207 2605-11209 2605-11241 2605-11242
2605-11264 2605-11265 2605-11266 2605-11302 2605-11323 2605-11326 2605-11330 2605-11332 2605-11367 2605-11380
2605-11400 2605-11407 2605-11413 2605-11414 2605-11445 2605-11465 2605-11472 2605-11490 2605-11504 2605-11540
2605-11549 2605-11564 2605-11607 2605-11615 2605-11618 2605-11632 2605-11636 2605-11655 2605-11659 2605-11667
2605-11669 2605-11686 2605-11701 2605-11714 2605-11725 2605-11732 2605-11736 2605-11746 2605-11749 2605-11755
2605-11763 2605-11771 2605-11785 2605-11793 2605-11798 2605-11811 2605-11812 2605-11850 2605-11862 2605-11869
2605-11873 2605-11883 2605-11886 2605-11897 2605-11903 2605-11910 2605-11930 2605-11939 2605-11948 2605-11992
2605-12000 2605-12006 2605-12011 2605-12014 2605-12017 2605-12034 2605-12051 2605-12073 2605-12083 2605-12095
2605-12097 2605-12102 2605-12105 2605-12115 2605-12126 2605-12149 2605-12150 2605-12169 2605-12173 2605-12206
2605-12215 2605-12219 2605-12221 2605-12230 2605-12247 2605-12248 2605-12259 2605-12269 2605-12280 2605-12282
2605-12288 2605-12292 2605-12300 2605-12303 2605-12311 2605-12325 2605-17361 2605-17368 2605-17374 2605-17409
2605-17427 2605-17450 2605-17456 2605-17466 2605-17473 2605-17484 2605-17495 2605-17500 2605-17515 2605-17530
2605-17545 2605-17588 2605-17591 2605-17595 2605-17606 2605-17613 2605-17614 2605-17648 2605-17671 2605-17696
2605-17718 2605-17739 2605-17753 2605-17756 2605-17794 2605-17813 2605-17842 2605-17853 2605-17857 2605-17887
2605-17894 2605-17919 2605-17931 2605-17942 2605-17945 2605-17950 2605-17953 2605-17971 2605-17985 2605-18007
2605-18018 2605-18019 2605-18021 2605-18042 2605-18052 2605-18061 2605-18070 2605-18081 2605-18083 2605-18086
2605-18107 2605-18110 2605-18123 2605-18130 2605-18135 2605-18136 2605-18140 2605-18168 2605-18171 2605-18179
2605-18212 2605-18220 2605-18225 2605-18228 2605-18235 2605-18236 2605-18245 2605-18248 2605-18273 2605-18274
2605-18285 2605-18288 2605-18304 2605-18306 2605-18324 2605-18328 2605-18339 2605-18343 2605-18356 2605-18363
2605-18365 2605-18373 2605-18377 2605-18384 2605-18388 2605-18390 2605-18401 2605-18404 2605-18421 2605-18433
2605-18460 2605-18471 2605-18487 2605-18489 2605-18529 2605-18545 2605-18576 2605-18579 2605-18582 2605-18590
2605-18633 2605-18636 2605-18641 2605-18644 2605-18645 2605-18678 2605-18682 2605-18688 2605-18694 2605-18695
2605-18707 2605-18710 2605-18726 2605-18727 2605-18733 2605-18736 2605-18758 2605-18763 2605-18856 2605-18868
2605-18874 2605-18887 2605-18893 2605-18906 2605-18924 2605-18937 2605-18947 2605-18962 2605-18977 2605-18990
2605-18994 2605-18996 2605-19023 2605-19068 2605-19083 2605-19085 2605-19089 2605-19138 2605-19144 2605-19145
2605-19160 2605-19168 2605-19190 2605-19200 2605-19237 2605-19246 2605-19268 2605-19276 2605-19288 2605-19308
2605-19335 2605-19350 2605-19356 2605-19371 2605-19372 2605-19418 2605-19424 2605-19431 2605-19447 2605-19453
2605-19461 2605-19462 2605-19467 2605-19488 2605-19493 2605-19495 2605-19499 2605-19514 2605-19519 2605-19583
2605-19594 2605-19614 2605-19615 2605-19617 2605-19627 2605-19631 2605-19639 2605-19655 2605-19656 2605-19664
2605-19682 2605-19684 2605-19702 2605-19705 2605-19707 2605-19724 2605-19726 2605-19748 2605-19777 2605-19778
2605-19796 2605-19823 2605-19850 2605-19858 2605-19872 2605-19876 2605-19884 2605-19901 2605-19905 2605-19924
2605-19930 2605-19934 2605-19946 2605-19959 2605-19969 2605-19989 2605-19991 2605-19994 2605-20003 2605-20007
2605-20008 2605-20010 2605-20012 2605-20013 2605-20024 2605-20035 2605-20063 2605-20067 2605-20092 2605-20100
2605-20110 2605-20120 2605-20133 2605-20137 2605-20142 2605-20147 2605-20170 2605-20184 2605-20216 2605-20230
2605-20236 2605-20252 2605-20316 2605-20319 2605-20342 2605-20353 2605-20365 2605-20369 2605-20374 2605-20382
2605-20395 2605-20412 2605-20427 2605-20471 2605-20497 2605-20544 2605-20557 2605-20565 2605-20581 2605-20586
2605-20593 2605-20601 2605-20604 2605-20635 2605-20639 2605-20658 2605-20662 2605-20672 2605-20679 2605-20697
2605-20708 2605-20719 2605-20742 2605-20752 2605-20753 2605-20774 2605-20778 2605-20800 2605-20802 2605-20806
2605-20814 2605-20840 2605-20844 2605-20850 2605-20851 2605-20852 2605-20869 2605-20875 2605-20895 2605-20896
2605-20906 2605-20909 2605-20933 2605-20942 2605-20945 2605-20958 2605-20969 2605-20980 2605-21023 2605-21034
2605-21038 2605-21054 2605-21058 2605-21059 2605-21066 2605-21071 2605-21089 2605-21105 2605-21118 2605-21150
2605-21158 2605-21173 2605-21181 2605-21194 2605-21202 2605-21213 2605-21229 2605-21234 2605-21255 2605-21259
2605-21260 2605-21261 2605-21286 2605-21321 2605-21322 2605-21325 2605-21335 2605-21355 2605-21381 2605-21392
2605-21400 2605-21419 2605-21424 2605-21450 2605-21457 2605-21463 2605-21466 2605-21487 2605-21489 2605-21494
2605-21495 2605-21501 2605-21520 2605-21523 2605-21530 2605-21546 2605-21547 2605-21564 2605-21570 2605-21577
2605-21582 2605-21616 2605-21625 2605-21658 2605-21668 2605-21684 2605-21693 2605-21708 2605-21709 2605-21715
2605-21732 2605-21733 2605-21755 2605-21780 2605-21792 2605-21796 2605-21802 2605-21808 2605-21834 2605-21849
2605-21862 2605-21866 2605-21871 2605-21884 2605-21897 2605-21905 2605-21917 2605-21919 2605-21920 2605-21927
2605-21929 2605-21931 2605-21945 2605-21949 2605-21954 2605-21970 2605-21985 2605-21988 2605-21992 2605-21998
2605-22002 2605-22046 2605-22051 2605-22056 2605-22059 2605-22063 2605-22076 2605-22080 2605-22104 2605-22111
2605-22150 2605-22154 2605-22159 2605-22160 2605-22165 2605-22175 2605-22188 2605-22237 2605-22244 2605-22253
2605-22261 2605-22277 2605-22285 2605-22286 2605-22292 2605-22301 2605-22309 2605-22327 2605-22335 2609-32840
2609-32872 2609-32883 2609-32891 2609-32892 2609-32895 2609-32898 2609-32899 2609-32926 2609-32934 2609-32936
2609-32937 2609-32941 2609-32945 2609-32970 2609-32991 2609-33007 2609-33030 2609-33038 2609-33057 2609-33088
2609-33090 2609-33092 2609-33100 2609-33103 2609-33105 2609-33106 2609-33124 2609-33136 2609-33140 2609-33159
2609-33160 2609-33173 2609-33174 2609-33183 2609-33189 2609-33197 2609-33201 2609-33202 2609-33223 2609-33230
2609-33242 2609-33246 2609-33249 2609-33252 2609-33257 2609-33264 2609-33270 2609-33284 2609-33290 2609-33299
2609-33300 2609-33321 2609-33334 2609-33345 2609-33351 2609-33354 2609-33375 2609-33387 2609-33395 2609-33418
2609-33419 2609-33430 2609-33437 2609-33439 2609-33457 2609-33459 2609-33463 2609-33470 2609-33474 2609-33481
2609-33495 2609-33505 2609-33539 2609-33541 2609-33558 2609-33561 2609-33568 2609-33581 2609-33586 2609-33588
2609-33590 2609-33602 2609-33611 2609-33613 2609-33646 2609-33661 2609-33679 2609-33681 2609-33708 2609-33728
2609-33731 2609-33732 2609-33733 2609-33751 2609-33756 2609-33759 2609-33765 2609-33784 2609-33789 2609-33790
2609-33796 2609-33807 2609-33812 2609-33828 2609-33833 2609-33839 2609-33848 2609-33854 2609-33856 2609-33871
2609-33875 2609-33900 2609-33907 2609-33919 2609-33941 2609-33944 2609-33949 2609-33954 2609-33967 2609-33971
2609-33979 2609-33994 2609-34032 2609-34043 2609-34048 2609-34057 2609-34059 2609-34086 2609-34090 2609-34097
2609-34115 2609-34130 2609-34132 2609-34139 2609-34154 2609-34169 2609-34172 2609-34174 2609-34175 2609-34197
2609-34225 2609-34234 2609-34258 2609-34260 2609-34262 2609-34272 2609-34279 2609-34307 2609-34328 2609-34330
2609-34352 2609-34362 2609-34374 2609-34388 2609-34421 2609-34423 2609-34432 2609-34435 2609-34440 2609-34444
2609-34471 2609-34477 2609-34482 2609-34483 2609-34497 2609-34500 2609-34516 2609-34517 2609-34551 2609-34557
2609-34563 2609-34586 2609-34588 2609-34595 2609-34597 2609-34603 2609-34604 2609-34610 2609-34616 2609-34635
2609-34636 2609-34654 2609-34665 2609-34669 2609-34684 2609-34690 2609-34694 2609-34700 2609-34731 2609-34732
2609-34746 2609-34754 2609-34760 2609-34797 2609-34801 2609-34821 2609-34838 2609-34845 2609-34855 2609-34856
2609-34859 2609-34878 2609-34879 2609-34887 2609-34914 2609-34922 2609-34925 2609-34927 2609-34928 2609-34937
2609-34957 2609-35006 2609-35013 2609-35015 2609-35018 2609-35033 2609-35047 2609-35054 2609-35071 2609-35080
2609-35088 2609-35108 2609-35114 2609-35134 2609-35143 2609-35157 2609-35181 2609-35187 2609-35190 2609-35192
2609-35198 2609-35203 2609-35206 2609-35232 2609-35263 2609-35276 2609-35300 2609-35303 2609-35318 2609-35320
2609-35329 2609-35331 2609-35348 2609-35350 2609-35363 2609-35374 2609-35387 2609-35396 2609-35402 2609-35403
2609-35406 2609-35472 2609-35474 2609-35489 2609-35491 2609-35520 2609-35539 2609-35543 2609-35594 2609-35602
2609-35606 2609-35617 2609-35621 2609-35636 2609-35644 2609-35652 2609-35671 2609-35681 2609-35705 2609-35722
2609-35727 2609-35737 2609-35738 2609-35740 2609-35741 2609-35745 2609-35747 2609-35761 2609-35762 2609-35766
2609-35768 2609-35771 2609-35778 2609-35780 2609-35790 2609-35798 2609-35814 2609-35832 2609-35836 2609-35850
2609-35853 2609-35865 2609-35873 2609-35893 2609-35897 2609-35902 2609-35907 2609-35916 2609-35917 2609-35927
2609-35932 2609-35950 2609-35963 2609-35973 2609-36002 2609-36012 2609-36016 2609-36029 2609-36048 2609-36049
2609-36055 2609-36078 2609-36093 2609-36094 2609-36097 2609-36113 2609-36166 2609-36171 2609-36177 2609-36183
2609-36210 2609-36225 2609-36230 2609-36235 2609-36236 2609-36238 2609-36240 2609-36252 2609-36262 2609-36282
2609-36302 2609-36303 2609-36313 2609-36315 2609-36335 2609-36341 2609-36375 2609-36386 2609-36393 2609-36413
2609-36421 2609-36434 2609-36438 2609-36447 2609-36499 2609-36504 2609-36521 2609-36526 2609-36527 2609-36529
2609-36546 2609-36564 2609-36569 2609-36574 2609-36602 2609-36612 2609-36613 2609-36614 2609-36616 2609-36636
2609-36644 2609-36683 2609-36715 2609-36727 2609-36730 2609-36738 2609-36757 2609-36772 2609-36777 2609-36788
2609-36793 2609-36794 2609-36798 2609-36802 2609-36803 2609-36812 2609-36819 2609-36821 2609-36837 2609-36838
2609-36846 2609-36877 2609-36885 2609-36891 2609-36898 2609-36904 2609-36923 2609-36934 2609-36944 2609-36956
2609-36978 2609-36979 2609-36989 2609-36997 2609-37005 2609-37025 2609-37026 2609-37038 2609-37045 2609-37080
2609-37086 2609-37101 2609-37104 2609-37112 2609-37114 2609-37115 2609-37118 2609-37119 2609-37124 2609-37131
2609-37134 2609-37147 2609-37151 2609-37156 2609-37165 2609-37173 2609-37181 2609-37218 2609-37223 2609-37225
2609-37265 2609-37271 2609-37273 2609-37282 2609-37313 2609-37334 2609-37340 2609-37345 2609-37367 2609-37373
2609-37375 2609-37377 2609-37398 2609-37414 2609-37427 2609-37430 2609-37436 2609-37438 2609-37446 2609-37450
2609-37484 2609-37487 2609-37519 2609-37535 2609-37568 2609-37573 2609-37575 2609-37580 2609-37588 2609-37593
2609-37595 2609-37615 2609-37621 2609-37633 2609-37648 2609-37654 2609-37660 2609-37673 2609-37674 2609-37679
2609-37685 2609-37686 2609-37704 2609-37708 2609-37712 2609-37714 2609-37725 2609-37726 2609-37733 2609-37758
2609-37775 2609-37784 2609-37785 2609-37798 2609-37808
""".split())


def carried_digest(doc):
    """SHA-256 of a data/parsed/<id>.json document without its parser version and the seconds of its versions."""
    doc = dict(doc)
    doc.pop("parser_version", None)
    doc["versions"] = [{k: v for k, v in r.items() if k != "seconds"} for r in doc["versions"]]
    return hashlib.sha256(json.dumps(doc, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()

In [ ]:
ID_FOLDER = re.compile(r"^\d{4}-\d{4,5}$")


def papers_to_parse():
    """{folder_id: number of versions} of every paper, in manifest order."""
    manifest = read_json(STATE_DIR / "discovery_manifest.json")
    with os.scandir(DATA_DIR) as entries:
        folders = {entry.name for entry in entries if entry.is_dir()}
    if manifest:
        records = manifest["papers"] if isinstance(manifest, dict) else manifest
        papers = {to_folder_id(p["id"]): int(p["versions"]) for p in records if p.get("versions", 0) > 0}
    else:
        papers = {}
        for name in sorted(folders):
            if ID_FOLDER.match(name):
                meta = read_json(DATA_DIR / name / "metadata.json")
                if meta and meta.get("versions"):
                    papers[name] = int(meta["versions"])
    extra = sorted(folders - set(papers))
    missing = [f for f in papers if f not in folders]
    print(f"{len(papers):,} papers ({sum(papers.values()):,} versions) | {len(folders):,} folders in data/raw"
          + (f" | not a paper: {', '.join(extra[:10])}" + (" ..." if len(extra) > 10 else "") if extra else "")
          + (f" | {len(missing):,} papers without a folder" if missing else ""))
    return papers


def read_done_index(path=DONE_INDEX, version=None):
    """{folder_id: summary} of the papers parsed with `version`, default PARSER_VERSION (last line wins)."""
    version = version or PARSER_VERSION
    done = {}
    try:
        with open(path, encoding="utf-8") as stream:
            for line in stream:
                try:
                    summary = json.loads(line)
                except ValueError:
                    continue   # a line torn by a killed session
                if summary.get("parser_version") == version:
                    done[summary["folder_id"]] = summary
                else:
                    done.pop(summary.get("folder_id"), None)
    except FileNotFoundError:
        pass
    return done


ALL_PAPERS = papers_to_parse()
if TEST_PAPERS:
    _ids = list(ALL_PAPERS)
    _step = max(1, len(_ids) // TEST_PAPERS)
    _chosen = dict.fromkeys(_ids[::_step][:TEST_PAPERS] + [f for f in HARD_CASES if f in ALL_PAPERS])
    PAPERS = {f: ALL_PAPERS[f] for f in _chosen}
    print(f"TEST_PAPERS: {len(PAPERS):,} papers ({sum(f in ALL_PAPERS for f in HARD_CASES)} hard cases)")
else:
    PAPERS = ALL_PAPERS

def read_failed_index(path=FAILED_INDEX):
    """{folder_id: error} of the papers that timed out or lost their worker with the current PARSER_VERSION."""
    failed = {}
    try:
        with open(path, encoding="utf-8") as stream:
            for line in stream:
                try:
                    row = json.loads(line)
                except ValueError:
                    continue   # a line torn by a killed session
                if row.get("parser_version") == PARSER_VERSION:
                    failed[row["folder_id"]] = row.get("error", "")
    except FileNotFoundError:
        pass
    return failed


CARRY_SAMPLE_FILE = M2_STATE / "carry_sample.json"   # {folder_id: carried_digest of its 13.7 file}


def carry_over():
    """Once: the CARRY_FROM summaries of the papers outside REPARSE_13_8 and the sample go to the index under
    PARSER_VERSION. Returns (carried, sample)."""
    current, old = read_done_index(), read_done_index(version=CARRY_FROM)
    candidates = sorted(f for f in old if f not in current and f not in REPARSE_13_8)
    sample = {}
    for folder_id in random.Random(13).sample(candidates, min(CARRY_SAMPLE, len(candidates))):
        try:
            sample[folder_id] = carried_digest(read_json(parsed_path(folder_id)))
        except (OSError, ValueError, TypeError, KeyError):
            continue   # no readable 13.7 file: parsed again like the others of the sample, not compared
    write_json(CARRY_SAMPLE_FILE, sample)
    carried = [f for f in candidates if f not in sample]
    with open(DONE_INDEX, "a", encoding="utf-8") as index:
        for folder_id in carried:
            index.write(json.dumps({**old[folder_id], "parser_version": PARSER_VERSION, "carried_from": CARRY_FROM}) + "\n")
    return carried, sample


if CARRY_OVER and CARRY_SAMPLE_FILE.exists():
    print(f"CARRY_OVER: done in an earlier session ({CARRY_SAMPLE_FILE.name})")
elif CARRY_OVER:
    if PARSER_VERSION != CARRY_TO:
        raise RuntimeError(f"CARRY_OVER is for {CARRY_FROM} -> {CARRY_TO}, not {PARSER_VERSION}: set it to False")
    _carried, _sample = carry_over()
    print(f"CARRY_OVER: {len(_carried):,} papers kept from {CARRY_FROM}; parsed again: the papers of REPARSE_13_8 "
          f"and a sample of {len(_sample)} others")

done = read_done_index()
failed = {} if RETRY_FAILED else {f: e for f, e in read_failed_index().items() if f in PAPERS and f not in done}
TODO = [f for f in PAPERS if f not in done and f not in failed]
print(f"{len(done.keys() & PAPERS.keys()):,} already parsed with this parser version, {len(TODO):,} to parse")
if failed:
    print(f"{len(failed):,} papers skipped: they timed out or lost their worker in an earlier session "
          f"(RETRY_FAILED = True parses them again): {', '.join(list(failed)[:10])}")

## 2. Parse in parallel, with the benchmark running

`parse_one` runs in a worker process: `parse_paper`, then `data/parsed/<id>.json`, outside the raw folders
of Milestone 1, which are only read (`write_json`: temporary file + rename; compact JSON, 25% smaller than indented, still about 1.5 × the
paper's `.tex` because JSON doubles every backslash), and a small summary back to the main process (counts,
no rows). The
main process alone writes the logs, so no two processes append to one file:

- `logs/m2_parse_status.csv`: `arxiv_id, version, status, bib_source, n_nodes, seconds, error`, one row per
  version. For an `empty_source` version the download receipts say why: `pdf_only` (arXiv has no TeX
  source), `not_found` (HTTP 404, e.g. a withdrawn version), `files_missing` (the receipt says the version
  was downloaded but its `vN/` folder is not there) or `not_downloaded`.
- `state/m2/done.jsonl`: the summary, written after `data/parsed/<id>.json`; a paper whose parsing failed is not in it
  and is parsed again next time.

**Time limit.** A paper gets `PAPER_TIMEOUT` seconds (600). Past it, it is logged as `error`
(`timeout: ...`), written to `state/m2/failed.jsonl` and skipped by the next sessions (`RETRY_FAILED = True`
parses those again). Two mechanisms enforce it:

- in the worker, `SIGALRM` (Linux) stops a paper that runs too long: an endless Python loop, and also a
  regular expression stuck in C code (both stopped after 5 s in the Colab test of 19 Nov,
  `16_test_m2_1000_papers`); the worker then goes on with the next paper;
- the workers are supervised: each takes one paper at a time, and a paper still without an answer
  `SUPERVISOR_GRACE` (60) seconds after its limit (its worker killed by the system, e.g. out of memory, which
  loses the task without any error, or stuck where the signal does not reach, e.g. in Drive I/O) is given
  up; the pool is then replaced and the other papers in flight are submitted again (a killed worker was
  given up after 15 s in the same test).

The pool is used whenever `fork` exists (Colab), also with `WORKERS = 1`. Without `fork` (Windows) the papers
are parsed in this process with no time limit.

**Benchmark.** RAM is sampled every 2 s as the RSS of this process **plus its worker processes**
(`psutil`, `children(recursive=True)`); the pool is started before the sampling thread. The session record
(`logs/m2_sessions/<id>.json`) is saved every 5 minutes and the samples appended to
`logs/m2_ram_timeseries.csv`, as in `05_run_pipeline`. A progress bar shows papers, versions, errors and the
ETA; a text line every 500 papers stays in the output after a browser reconnect.

In [ ]:
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None
if ON_COLAB and tqdm is not None:
    from tqdm.std import tqdm   # Colab: text bars, which survive a browser reconnect


class PaperTimeout(BaseException):
    """Raised by SIGALRM after PAPER_TIMEOUT s. Not an Exception: the per-version `except Exception` of
    parse_paper must not catch it and go on with the next version without a limit."""


def _alarm(signum, frame):
    raise PaperTimeout()


@contextlib.contextmanager
def time_limit(seconds):
    """PaperTimeout after `seconds`, with SIGALRM (Linux, main thread of the process). On Colab it stops an
    endless loop and a regular expression stuck in C (tested); the supervisor handles a worker that dies."""
    if not seconds or not hasattr(signal, "SIGALRM") or threading.current_thread() is not threading.main_thread():
        yield
        return
    previous = signal.signal(signal.SIGALRM, _alarm)
    signal.alarm(max(1, int(seconds)))
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, previous)


def failed_summary(folder_id, error, seconds=None, timed_out=False):
    """Summary of a paper that failed as a whole: every version of the manifest is an error. It reads
    nothing, since Drive may be what hangs."""
    error = error[:500]
    return {"folder_id": folder_id, "arxiv_id": folder_id.replace("-", ".", 1), "ok": False,
            "timed_out": timed_out, "error": error,
            "versions": [{"version": v, "status": "error", "bib_source": "", "n_nodes": 0, "seconds": 0.0,
                          "error": error} for v in range(1, (PAPERS.get(folder_id) or 1) + 1)],
            "seconds": None if seconds is None else round(seconds, 3)}


def parse_one(folder_id):
    """Worker: parse one paper and write data/parsed/<id>.json; returns a summary without the rows."""
    pdir = DATA_DIR / folder_id
    started = time.perf_counter()
    try:
        with time_limit(PAPER_TIMEOUT):
            result = parse_paper(pdir)
            write_json(parsed_path(folder_id), to_parsed_json(result), indent=None)   # compact JSON
            return {
                "folder_id": folder_id, "arxiv_id": result["arxiv_id"], "parser_version": result["parser_version"],
                "ok": True, "versions": result["versions"],
                "nodes_by_kind": result["counts"]["nodes_by_kind"],
                "elements_by_kind": dict(Counter(row["kind"] for row in result["elements"])),
                "bib_before": result["counts"]["bib_entries_before"], "bib_after": len(result["bibtex"]),
                "rows": {table: len(result[table]) for table in ("elements", "hierarchy", "bibtex")},
                "bytes": parsed_path(folder_id).stat().st_size,
                "problems": check_paper(result, len(result["versions"])),
                "seconds": round(time.perf_counter() - started, 3),
            }
    except PaperTimeout:
        return failed_summary(folder_id, f"timeout: still parsing after {PAPER_TIMEOUT} s",
                              time.perf_counter() - started, timed_out=True)
    except Exception as err:   # the whole paper failed (e.g. no metadata.json): every version is an error
        return failed_summary(folder_id, _error_text(err), time.perf_counter() - started)


def empty_reasons():
    """{(arxiv_id, version): reason} for versions without source, from the download receipts and errors."""
    # receipt "ok" but no vN/ folder: the files never reached Drive (e.g. a session killed before Drive synced)
    reasons = {key: "pdf_only" if receipt.get("status") == "empty_source" else "files_missing"
               for key, receipt in _read_receipt_index(DOWNLOAD_STATE).items()}
    try:
        with open(DOWNLOAD_STATE / "errors.jsonl", encoding="utf-8") as stream:
            for line in stream:
                try:
                    row = json.loads(line)
                except ValueError:
                    continue
                key = (row.get("arxiv_id"), row.get("version"))
                if key not in reasons and (row.get("error_kind") == "NOT_FOUND" or "HTTP 404" in row.get("error", "")):
                    reasons[key] = "not_found"
    except FileNotFoundError:
        pass
    return reasons


def cpu_info():
    model = platform.processor() or ""
    try:
        with open("/proc/cpuinfo", encoding="utf-8") as stream:
            model = next((l.split(":", 1)[1].strip() for l in stream if l.startswith("model name")), model)
    except OSError:
        pass
    return f"{model} x {os.cpu_count()}".strip()


class M2Monitor:
    """RSS of this process and of all its children, every `interval` seconds, in a background thread."""

    def __init__(self, interval=2.0):
        self.interval, self.samples = interval, []   # samples: (unix time, MB)
        self.start_time = self.end_time = None
        self._process = psutil.Process()
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._loop, daemon=True)

    def rss_mb(self):
        total = self._process.memory_info().rss
        for child in self._process.children(recursive=True):
            try:
                total += child.memory_info().rss
            except psutil.Error:
                pass   # a worker that just ended
        return total / MB

    def _loop(self):
        while not self._stop.is_set():
            try:
                self.samples.append((time.time(), self.rss_mb()))
            except Exception:
                pass
            self._stop.wait(self.interval)

    def start(self):
        self.start_time = time.time()
        self._thread.start()
        return self

    def stop(self):
        if self.end_time is None:
            self.end_time = time.time()
        self._stop.set()
        self._thread.join(timeout=5)


class M2Session:
    """This session in logs/m2_sessions/<id>.json, saved every `every` seconds (complete: false until the end)."""

    def __init__(self, monitor, every=300):
        self.monitor, self.every = monitor, every
        started = datetime.fromtimestamp(monitor.start_time, tz=timezone.utc)
        self.id = started.strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
        self.path = M2_SESSIONS / f"{self.id}.json"
        self.papers = self.versions = 0
        self.flushed = 0
        self._guard = threading.Lock()
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def _loop(self):
        while not self._stop.wait(self.every):
            try:
                self.save()
            except Exception as err:   # a full Drive must not stop the parsing
                print(f"[benchmark] could not save this session: {err!r}")

    def save(self, final=None):
        with self._guard:
            monitor = self.monitor
            end = monitor.end_time or time.time()
            samples = list(monitor.samples)
            rams = [mb for _, mb in samples]
            record = {
                "session": self.id, "complete": final is not None,
                "start": to_iso(datetime.fromtimestamp(monitor.start_time, tz=timezone.utc)),
                "end": to_iso(datetime.fromtimestamp(end, tz=timezone.utc)),
                "wall_seconds": round(end - monitor.start_time, 3),
                "papers": self.papers, "versions": self.versions, "workers": WORKERS,
                "ram_peak_mb": round(max(rams, default=0.0), 2),
                "ram_avg_mb": round(sum(rams) / len(rams), 2) if rams else None,
                "ram_samples": len(rams), "parser_version": PARSER_VERSION,
            }
            record.update(final or {})
            M2_SESSIONS.mkdir(parents=True, exist_ok=True)
            write_json(self.path, record)
            new = samples[self.flushed:]
            if new:
                header = not M2_RAM_LOG.exists()
                with M2_RAM_LOG.open("a", newline="", encoding="utf-8") as stream:
                    writer = csv.writer(stream)
                    if header:
                        writer.writerow(["utc_time", "rss_mb"])
                    for moment, mb in new:
                        writer.writerow([to_iso(datetime.fromtimestamp(moment, tz=timezone.utc)), round(mb, 2)])
                self.flushed = len(samples)
            return record

    def finish(self, final):
        self._stop.set()
        self._thread.join(timeout=5)
        return self.save(final=final)


def stop_pool(pool):
    """terminate() in a thread, at most 30 s: joining a worker stuck in Drive I/O could block for ever."""
    stopper = threading.Thread(target=lambda: (pool.terminate(), pool.join()), daemon=True)
    stopper.start()
    stopper.join(30)


SUPERVISOR_GRACE = 60   # seconds past PAPER_TIMEOUT before a silent worker is given up


def supervised(pool, context, todo):
    """Summaries of `todo` from WORKERS processes, as they finish; see "Time limit" above."""
    limit = (PAPER_TIMEOUT or 3600) + SUPERVISOR_GRACE
    queue, running = list(reversed(todo)), {}   # running: folder_id -> (AsyncResult, start time)
    ready = threading.Event()

    def wake(_):   # called by the pool's result thread
        ready.set()

    try:
        while queue or running:
            out = []
            for folder_id in [f for f, (job, _) in running.items() if job.ready()]:
                job, started = running.pop(folder_id)
                try:
                    out.append(job.get())
                except Exception as err:   # parse_one returns its own errors: this is the pool's
                    out.append(failed_summary(folder_id, "worker error: " + _error_text(err), time.monotonic() - started))
            now = time.monotonic()
            late = [f for f, (_, started) in running.items() if now - started > limit]
            for folder_id in late:
                running.pop(folder_id)
                out.append(failed_summary(folder_id, f"timeout: no answer from its worker after {limit} s "
                                                     "(stuck, or killed by the system: out of memory?)",
                                          limit, timed_out=True))
            if late:   # which worker is stuck or dead cannot be told: replace the pool
                stop_pool(pool)
                queue.extend(running)
                running.clear()
                pool = context.Pool(WORKERS)
            while queue and len(running) < WORKERS:   # one paper per worker, so its clock starts now
                folder_id = queue.pop()
                running[folder_id] = (pool.apply_async(parse_one, (folder_id,), callback=wake, error_callback=wake),
                                      time.monotonic())
            yield from out
            if running and not out:
                ready.wait(1.0)   # a worker finished, or time to look for late papers again
            ready.clear()
    finally:
        stop_pool(pool)


def run_parsing(todo):
    """Parse `todo` (folder IDs) with WORKERS processes; logs and index are written as results arrive."""
    reasons = empty_reasons()
    # Supervised workers whenever fork exists (Colab), also with WORKERS = 1: a worker that dies (out of
    # memory) or hangs where SIGALRM does not reach can only be handled from another process.
    use_fork = "fork" in multiprocessing.get_all_start_methods()
    context = multiprocessing.get_context("fork") if use_fork and todo else None
    pool = context.Pool(WORKERS) if context else None   # before any thread
    results = None
    lock = RunLock(M2_STATE / "parse.lock").acquire()
    monitor = M2Monitor().start()
    session = M2Session(monitor)
    counts = Counter()
    new_log = not STATUS_LOG.exists()
    bar = tqdm(total=len(todo), unit="paper", desc="Parsing", dynamic_ncols=True, mininterval=2) \
        if tqdm and todo else None
    started = time.perf_counter()
    try:
        with (open(STATUS_LOG, "a", newline="", encoding="utf-8") as log,
              open(DONE_INDEX, "a", encoding="utf-8") as index,
              open(FAILED_INDEX, "a", encoding="utf-8") as failed_log):
            writer = csv.DictWriter(log, fieldnames=STATUS_FIELDS, extrasaction="ignore")
            if new_log:
                writer.writeheader()
            results = supervised(pool, context, todo) if pool else map(parse_one, todo)
            for n, summary in enumerate(results, 1):
                for record in summary["versions"]:
                    row = dict(record, arxiv_id=summary["arxiv_id"])
                    if row["status"] == "empty_source":
                        row["error"] = reasons.get((summary["arxiv_id"], row["version"]), "not_downloaded")
                    writer.writerow(row)
                    counts[row["status"]] += 1
                log.flush()
                if summary["ok"]:
                    index.write(json.dumps(summary, ensure_ascii=False) + "\n")
                    index.flush()
                else:
                    counts["paper_error"] += 1
                if summary.get("timed_out"):   # skipped by the next sessions unless RETRY_FAILED
                    counts["timeout"] += 1
                    failed_log.write(json.dumps({"folder_id": summary["folder_id"], "parser_version": PARSER_VERSION,
                                                 "error": summary["error"],
                                                 "utc": to_iso(datetime.now(timezone.utc))}) + "\n")
                    failed_log.flush()
                    print(f"[{datetime.now().strftime('%H:%M:%S')}] {summary['folder_id']}: {summary['error']}",
                          flush=True)
                counts["problems"] += bool(summary.get("problems"))
                session.papers += 1
                session.versions += len(summary["versions"])
                if bar is not None:
                    bar.update(1)
                    bar.set_postfix_str(f"versions ok {counts['ok']:,} | empty {counts['empty_source']:,} | "
                                        f"no root {counts['no_root']:,} | errors {counts['error']:,} | "
                                        f"timeouts {counts['timeout']:,}")
                if n % 500 == 0 or n == len(todo):
                    elapsed = time.perf_counter() - started
                    eta = elapsed / n * (len(todo) - n)
                    print(f"[{datetime.now().strftime('%H:%M:%S')}] {n:,}/{len(todo):,} papers | "
                          f"{elapsed / n:.2f} s/paper | ETA {eta / 60:,.1f} min | {dict(counts)}", flush=True)
    except KeyboardInterrupt:
        print("Interrupted: finished papers are saved; run all cells again to continue.")
    finally:
        if bar is not None:
            bar.close()
        if pool is not None:
            if results is not None:
                results.close()   # the supervisor stops the pool it uses (it may have replaced this one)
            stop_pool(pool)
        lock.release()
    return monitor, session, counts


monitor, session, run_counts = run_parsing(TODO)

## 3. The three tables

`merge_tables` reads `data/parsed/<id>.json` of the papers of this run (with the current parser version), 500 papers
at a time with 16 threads (Drive reads), and appends each batch to the three Parquet files with
`pyarrow.parquet.ParquetWriter`, so memory holds one batch only. The files are written on the local disk
and copied to `data/m2/` at the end. Types follow the statement: `kind` and `entry_type` are
dictionary-encoded strings, `version` and `year` integers (`year` nullable), `authors` a list of strings,
`element_id`, `arxiv_id`, `parent_id` and `citation_key` non-null.

The same pass checks the invariants on every batch (a paper's rows are always in one batch): no duplicate
`element_id`; every `element_id` and `parent_id` of `hierarchy` is in `elements`; in each (paper, version)
the only parent that is never a child is the document root.

In [ ]:
SCHEMAS = {
    "elements": pa.schema([
        pa.field("element_id", pa.string(), nullable=False), pa.field("arxiv_id", pa.string(), nullable=False),
        pa.field("kind", pa.dictionary(pa.int32(), pa.string())), pa.field("content", pa.string())]),
    "hierarchy": pa.schema([
        pa.field("arxiv_id", pa.string(), nullable=False), pa.field("version", pa.int32()),
        pa.field("element_id", pa.string(), nullable=False), pa.field("parent_id", pa.string(), nullable=False)]),
    "bibtex": pa.schema([
        pa.field("arxiv_id", pa.string(), nullable=False), pa.field("citation_key", pa.string(), nullable=False),
        pa.field("entry_type", pa.dictionary(pa.int32(), pa.string())), pa.field("title", pa.string()),
        pa.field("authors", pa.list_(pa.string())), pa.field("year", pa.int32()), pa.field("doi", pa.string()),
        pa.field("journal", pa.string()), pa.field("url", pa.string())]),
}


def check_batch(elements, hierarchy):
    """Broken invariants of a batch of papers (columns of elements and hierarchy)."""
    problems = []
    ids = elements["element_id"]
    known = set(ids)
    if len(known) != len(ids):
        problems.append(f"{len(ids) - len(known)} duplicate element_id")
    parents, children = defaultdict(set), defaultdict(set)
    for arxiv_id, version, element_id, parent_id in zip(*(hierarchy[c] for c in HIERARCHY_COLUMNS)):
        if element_id not in known or parent_id is None or parent_id not in known:
            problems.append(f"{arxiv_id} v{version}: edge {element_id} -> {parent_id} outside elements")
        parents[(arxiv_id, version)].add(parent_id)
        children[(arxiv_id, version)].add(element_id)
    for (arxiv_id, version), parent_ids in parents.items():
        roots = parent_ids - children[(arxiv_id, version)]
        if roots != {to_folder_id(arxiv_id) + "#doc"}:
            problems.append(f"{arxiv_id} v{version}: roots {sorted(roots)}")
    return problems


def read_parsed(folder_id):
    """data/parsed/<id>.json, or None if it is missing or unreadable (a file torn by a Drive disconnect)."""
    try:
        return read_json(parsed_path(folder_id))
    except (OSError, ValueError):
        return None


def merge_tables(folder_ids, out_dir=M2_DATA, batch=500, threads=16):
    """The three Parquet tables of the given papers; returns ({table: rows}, problems, bytes written,
    papers whose file could not be read)."""
    local = Path(tempfile.mkdtemp(prefix="m2-tables-"))
    writers = {name: pq.ParquetWriter(local / f"{name}.parquet", schema, compression="zstd")
               for name, schema in SCHEMAS.items()}
    rows, problems, unreadable = Counter(), [], []
    bar = tqdm(total=len(folder_ids), unit="paper", desc="Tables", dynamic_ncols=True) if tqdm else None
    readers = ThreadPoolExecutor(threads)
    try:
        for start in range(0, len(folder_ids), batch):
            chunk = folder_ids[start:start + batch]
            try:   # Drive may stop answering: never wait for ever
                loaded = list(readers.map(read_parsed, chunk, timeout=READ_TIMEOUT))
            except TimeoutError:
                raise RuntimeError(f"reading {len(chunk)} files of data/parsed/ took more than {READ_TIMEOUT} s: "
                                   "is Google Drive still mounted? Run all cells again.") from None
            unreadable += [f for f, doc in zip(chunk, loaded) if doc is None]
            docs = [doc for doc in loaded if doc is not None]
            columns = {}
            for name, schema in SCHEMAS.items():
                columns[name] = {field: [v for d in docs for v in d[name][field]] for field in schema.names}
                writers[name].write_table(pa.Table.from_pydict(columns[name], schema=schema))
                rows[name] += len(columns[name][schema.names[0]])
            problems += check_batch(columns["elements"], columns["hierarchy"])
            if bar is not None:
                bar.update(len(chunk))
    finally:
        readers.shutdown(wait=False, cancel_futures=True)   # a thread stuck on Drive is left behind
        for writer in writers.values():
            writer.close()
        if bar is not None:
            bar.close()
    out_dir.mkdir(parents=True, exist_ok=True)
    written = 0
    for name in SCHEMAS:
        target = out_dir / f"{name}.parquet"
        shutil.copyfile(local / f"{name}.parquet", target.with_suffix(".parquet.tmp"))
        os.replace(target.with_suffix(".parquet.tmp"), target)
        written += target.stat().st_size
    shutil.rmtree(local, ignore_errors=True)
    return dict(rows), problems, written, unreadable


def check_carry_sample():
    """Section 1: the sample of papers outside REPARSE_13_8, parsed again, must give their 13.7 files."""
    sample = read_json(CARRY_SAMPLE_FILE) if CARRY_SAMPLE_FILE.exists() else None
    if not sample:
        return []
    parsed = [f for f in sample if f in done]
    differ = [f for f in parsed if (read_parsed(f) is None or carried_digest(read_parsed(f)) != sample[f])]
    print(f"CARRY_OVER sample: {len(parsed) - len(differ)} of {len(parsed)} papers parsed again give the same "
          f"file as {CARRY_FROM}" + (f" ({len(sample) - len(parsed)} not parsed yet)" if len(parsed) < len(sample) else ""))
    if differ:
        return [f"CARRY_OVER sample: {len(differ)} papers outside REPARSE_13_8 changed ({', '.join(differ[:10])}): "
                "the carried papers may be stale, run 15 again with FRESH_RUN = True"]
    return []


done = read_done_index()
in_run = [f for f in PAPERS if f in done]
table_rows, table_problems, parquet_bytes = {}, [], 0
if MERGE_TABLES and in_run:
    table_rows, table_problems, parquet_bytes, unreadable = merge_tables(in_run)
    if unreadable:   # parsed again next session: a line with another parser version takes a paper out of done
        with open(DONE_INDEX, "a", encoding="utf-8") as index:
            for folder_id in unreadable:
                index.write(json.dumps({"folder_id": folder_id, "parser_version": None}) + "\n")
        table_problems.append(f"{len(unreadable)} unreadable files in data/parsed/, parsed again next session: "
                              + ", ".join(unreadable[:10]))
    expected = {t: sum(done[f]["rows"][t] for f in in_run if f not in unreadable) for t in SCHEMAS}
    print(f"Tables in {M2_DATA}: " + ", ".join(f"{t} {n:,} rows" for t, n in table_rows.items())
          + f" | {parquet_bytes / MB:,.1f} MB")
    if table_rows != expected:
        table_problems.append(f"row counts {table_rows} differ from the parsed papers {expected}")
table_problems += check_carry_sample()

## 4. Checks, statistics and benchmark

**Checks** (printed, and `AssertionError` if one fails): the invariants of section 3; `check_paper` on every
paper (well-formed IDs, unique, parents present, one root per version); every version of every paper of the
run is in `logs/m2_parse_status.csv`.

**`logs/m2_stats.json`** (for the report, all sessions; papers of this run):
- parse success rate: `ok` versions over the versions with a `.tex` file, and over all versions;
- version coverage per paper: versions parsed / versions in `metadata.json`, as a distribution (`"2/3": n`)
  and summary counts;
- elements before dedup (every node of every version) and after (rows of `elements`), per `kind`;
- BibTeX entries before dedup (summed over versions) and after (rows of `bibtex`);
- distributions of `bib_source` and `status`.

**`logs/m2_benchmark.json`** (all sessions): `start`, `end`, `wall_seconds` (sum of the sessions),
`papers`, `versions`, `avg_seconds_per_paper` (wall time / papers), `peak_ram_mb` (max), `avg_ram_mb`
(time-weighted), `output_bytes` (`data/parsed/` + Parquet tables), `ram_timeseries` (at most 5,000
`[seconds, MB]` points), `workers`, `colab_cpu`.

In [ ]:
def status_rows():
    """{(arxiv_id, version): last row} of logs/m2_parse_status.csv."""
    rows = {}
    if STATUS_LOG.exists():
        with open(STATUS_LOG, newline="", encoding="utf-8") as stream:
            for row in csv.DictReader(stream):
                try:
                    rows[(row["arxiv_id"], int(row["version"]))] = row
                except (KeyError, TypeError, ValueError):
                    continue
    return rows


def compute_stats(papers, done, logged):
    versions, coverage, nodes, elements, bib_before, bib_after = [], Counter(), Counter(), Counter(), 0, 0
    for folder_id, n_versions in papers.items():
        arxiv_id = folder_id.replace("-", ".", 1)
        summary = done.get(folder_id)
        if summary:
            records = summary["versions"]
            nodes.update(summary["nodes_by_kind"])
            elements.update(summary["elements_by_kind"])
            bib_before += summary["bib_before"]
            bib_after += summary["bib_after"]
        else:   # not parsed (the paper failed): its versions as logged, else as errors
            records = [logged.get((arxiv_id, v)) or {"status": "error", "bib_source": ""} for v in range(1, n_versions + 1)]
        versions += records
        coverage[f"{sum(r['status'] == 'ok' for r in records)}/{n_versions}"] += 1
    status = Counter(r["status"] for r in versions)
    with_tex = sum(status[s] for s in ("ok", "no_root", "error"))
    share = lambda part, whole: round(100 * part / whole, 2) if whole else None   # noqa: E731
    full = sum(n for key, n in coverage.items() if key.split("/")[0] == key.split("/")[1])
    none = sum(n for key, n in coverage.items() if key.startswith("0/"))
    return {
        "computed_at": to_iso(datetime.now(timezone.utc)), "parser_version": PARSER_VERSION,
        "implementation": IMPLEMENTATION,
        "papers": len(papers), "papers_parsed": sum(f in done for f in papers),
        "versions": len(versions), "versions_with_tex": with_tex,
        "parse_success_rate_of_tex_versions_pct": share(status["ok"], with_tex),
        "parse_success_rate_of_all_versions_pct": share(status["ok"], len(versions)),
        "version_coverage": {
            "papers_full": full, "papers_partial": len(papers) - full - none, "papers_none": none,
            "mean_pct": round(statistics.mean(100 * int(k.split("/")[0]) / int(k.split("/")[1])
                                              for k, n in coverage.items() for _ in range(n)), 2) if coverage else None,
            "distribution": dict(sorted(coverage.items(), key=lambda kv: (int(kv[0].split("/")[1]), kv[0]))),
        },
        "elements_before_dedup": sum(nodes.values()), "elements_after_dedup": sum(elements.values()),
        "elements_before_dedup_by_kind": dict(nodes.most_common()),
        "elements_after_dedup_by_kind": dict(elements.most_common()),
        "bib_entries_before_dedup": bib_before, "bib_entries_after_dedup": bib_after,
        "status": dict(status.most_common()),
        "bib_source_of_tex_versions": dict(Counter(r["bib_source"] for r in versions if r["status"] != "empty_source").most_common()),
        "empty_source_reasons": dict(Counter(logged.get((p.replace("-", ".", 1), v), {}).get("error", "")
                                             for p, n in papers.items() for v in range(1, n + 1)
                                             if (logged.get((p.replace("-", ".", 1), v)) or {}).get("status") == "empty_source").most_common()),
    }


def load_m2_sessions():
    records = [read_json(p) for p in M2_SESSIONS.glob("*.json")] if M2_SESSIONS.is_dir() else []
    return sorted((r for r in records if r), key=lambda r: r["start"])


def compute_benchmark(papers, done, output_bytes):
    sessions = load_m2_sessions()
    wall = sum(s["wall_seconds"] for s in sessions)
    parsed = [done[f] for f in papers if f in done]
    series = []
    if M2_RAM_LOG.exists():
        with open(M2_RAM_LOG, newline="", encoding="utf-8") as stream:
            samples = list(csv.DictReader(stream))
        if samples:
            first = datetime.fromisoformat(samples[0]["utc_time"])
            step = max(1, len(samples) // 5000)
            series = [[round((datetime.fromisoformat(s["utc_time"]) - first).total_seconds(), 1), float(s["rss_mb"])]
                      for s in samples[::step]]
    return {
        "start": sessions[0]["start"] if sessions else None, "end": sessions[-1]["end"] if sessions else None,
        "wall_seconds": round(wall, 1), "sessions": len(sessions),
        "papers": len(parsed), "versions": sum(len(p["versions"]) for p in parsed),
        "avg_seconds_per_paper": round(wall / len(parsed), 3) if parsed else None,
        "median_paper_seconds_in_worker": round(statistics.median(p["seconds"] for p in parsed), 3) if parsed else None,
        "peak_ram_mb": max((s["ram_peak_mb"] for s in sessions), default=None),
        "avg_ram_mb": round(sum((s["ram_avg_mb"] or 0) * s["wall_seconds"] for s in sessions) / wall, 2) if wall else None,
        "output_bytes": output_bytes,
        "ram_timeseries": series,
        "workers": WORKERS, "colab_cpu": cpu_info(), "parser_version": PARSER_VERSION,
    }


done = read_done_index()
logged = status_rows()

# Checks
paper_problems = {f: done[f]["problems"] for f in PAPERS if f in done and done[f].get("problems")}
missing_rows = [(f, v) for f, n in PAPERS.items() for v in range(1, n + 1)
                if (f.replace("-", ".", 1), v) not in logged]
failed_papers = [f for f in PAPERS if f not in done]
print(f"Checks: {len(paper_problems)} papers with broken invariants, {len(table_problems)} table problems, "
      f"{len(missing_rows)} versions missing from {STATUS_LOG.name}, {len(failed_papers)} papers not parsed")
given_up = {f: e for f, e in read_failed_index().items() if f in PAPERS and f not in done}
if given_up:
    print(f"   {len(given_up)} of them timed out or lost their worker (state/m2/failed.jsonl, "
          f"RETRY_FAILED = True to try again): " + "; ".join(f"{f} {e[:60]}" for f, e in list(given_up.items())[:5]))
for folder_id, problems in list(paper_problems.items())[:5]:
    print("  ", folder_id, problems[:3])
for problem in table_problems[:5]:
    print("  ", problem)

# Statistics
stats = compute_stats(PAPERS, done, logged)
write_json(M2_STATS, stats)

# Benchmark: this session's final record, then every session
monitor.stop()
parsed_bytes = sum(done[f]["bytes"] for f in PAPERS if f in done)
final = session.finish({"output_bytes": parsed_bytes + parquet_bytes, "parquet_bytes": parquet_bytes})
M2_SESSIONS_LOG.write_text("".join(json.dumps(s) + "\n" for s in load_m2_sessions()), encoding="utf-8")
benchmark = compute_benchmark(PAPERS, done, parsed_bytes + parquet_bytes)
write_json(M2_BENCHMARK, benchmark)

b, s = benchmark, stats
lines = [
    ("Papers parsed / in this run", f"{s['papers_parsed']:,} / {s['papers']:,}"),
    ("Versions: ok / empty_source / no_root / error", " / ".join(f"{s['status'].get(k, 0):,}" for k in VERSION_STATUSES)),
    ("Parse success rate: of versions with .tex / of all", f"{s['parse_success_rate_of_tex_versions_pct']}% / "
                                                          f"{s['parse_success_rate_of_all_versions_pct']}%"),
    ("Papers with every version parsed / some / none", f"{s['version_coverage']['papers_full']:,} / "
                                                       f"{s['version_coverage']['papers_partial']:,} / {s['version_coverage']['papers_none']:,}"),
    ("Elements before -> after dedup", f"{s['elements_before_dedup']:,} -> {s['elements_after_dedup']:,}"),
    ("BibTeX entries before -> after dedup", f"{s['bib_entries_before_dedup']:,} -> {s['bib_entries_after_dedup']:,}"),
    ("bib_source (versions with .tex)", ", ".join(f"{k} {v:,}" for k, v in s["bib_source_of_tex_versions"].items())),
    ("Wall time (all sessions) / sessions", f"{b['wall_seconds'] / 60:,.1f} min / {b['sessions']}"),
    ("Average time per paper (wall / papers)", f"{b['avg_seconds_per_paper']} s with {WORKERS} workers"),
    ("Peak RAM / average RAM (with workers)", f"{b['peak_ram_mb']} MB / {b['avg_ram_mb']} MB"),
    ("Output: data/parsed + Parquet", f"{parsed_bytes / MB:,.1f} MB + {parquet_bytes / MB:,.1f} MB"),
    ("Estimate for all papers", f"{b['avg_seconds_per_paper'] * len(ALL_PAPERS) / 3600:,.1f} h for {len(ALL_PAPERS):,} papers"
                                if b["avg_seconds_per_paper"] else "-"),
]
width = max(len(name) for name, _ in lines)
print("\n" + "\n".join(f"{name:<{width}}  {value}" for name, value in lines))
print(f"\nSaved {M2_STATS.name}, {M2_BENCHMARK.name}, {M2_SESSIONS_LOG.name}, {M2_RAM_LOG.name}, {STATUS_LOG.name}")

if IN_COLAB and FLUSH_DRIVE_AT_END:
    from google.colab import drive
    drive.flush_and_unmount()   # Drive writes are asynchronous: make sure every file reached Drive
    print("Google Drive flushed and unmounted: every file of this session is saved. Run all mounts it again.")
assert not (paper_problems or table_problems or missing_rows), "M2 checks failed (see above)"